In [ ]:
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
import matplotlib.pyplot as plt  # For histogram

# --- Configuration ---
folder_path = Path("../data/jsonl_stage1")   # 🔵 Change to your folder
model_name = "google/gemma-3-4b-it"   # 🔵 Your model

# --- Load tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# --- Find all .jsonl files ---
jsonl_files = list(folder_path.glob("*.jsonl"))
print(f"Found {len(jsonl_files)} JSONL files.")

overall_max_len = 0
file_max_lengths = {}
all_token_counts = []  # 🔵 NEW: collect token counts across all files

# --- Loop over files ---
for jsonl_path in jsonl_files:
    print(f"Processing {jsonl_path.name}...")
    dataset = load_dataset("json", data_files=str(jsonl_path), split="train", cache_dir=".cache")
    
    max_len = 0
    for example in dataset:
        text = example["input"]
        tokens = tokenizer(text, truncation=False, padding=False)["input_ids"]
        n_tokens = len(tokens)
        all_token_counts.append(n_tokens)  # 🔵 Collect all token counts
        if n_tokens > max_len:
            max_len = n_tokens
    file_max_lengths[jsonl_path.name] = max_len
    
    if max_len > overall_max_len:
        overall_max_len = max_len

# --- Print results ---
print("\n📊 Max token counts per file:")
for file, length in file_max_lengths.items():
    print(f"{file}: {length} tokens")

print(f"\n📏 Overall maximum token count: {overall_max_len} tokens")

# --- 🔵 NEW: Compute and print distribution statistics ---
all_token_counts_np = np.array(all_token_counts)

print("\n📈 Token count distribution:")
print(f"Min: {all_token_counts_np.min()}")
print(f"Max: {all_token_counts_np.max()}")
print(f"Mean: {all_token_counts_np.mean():.2f}")
print(f"Median: {np.median(all_token_counts_np)}")
print(f"Std Dev: {all_token_counts_np.std():.2f}")
print(f"25th percentile: {np.percentile(all_token_counts_np, 25)}")
print(f"75th percentile: {np.percentile(all_token_counts_np, 75)}")

# --- 🔵 NEW: Plot histogram ---
plt.figure(figsize=(10, 6))
plt.hist(all_token_counts_np, bins=50, edgecolor='black')
plt.title("Distribution of Token Counts")
plt.xlabel("Number of Tokens")
plt.ylabel("Number of Examples")
plt.grid(True)
plt.show()

count_above_16k = np.sum(all_token_counts_np > 16000)
percentage_above_16k = (count_above_16k / len(all_token_counts_np)) * 100

print(f"\n🔵 {count_above_16k} examples ({percentage_above_16k:.2f}%) have more than 16,000 tokens.")

Found 18 JSONL files.
Processing train_financial_commercial_terms.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

Processing train_competition_exclusivity.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

Processing val_termination_control_rights.jsonl...


...

Generating train split: 0 examples [00:00, ? examples/s]

Processing train_termination_control_rights.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

Processing val_financial_commercial_terms.jsonl...


Generating train split: 0 examples [00:00, ? examples/s]

In [10]:
from transformers import AutoTokenizer
import json

tok = AutoTokenizer.from_pretrained("google/gemma-3-4b-it")
ex = json.loads(open("../data/spans/spans_train_filenames.jsonl").readline())
text = ex["text"]
for sp in ex["clauses"]:
    print(text[sp["start"]:sp["end"]+1])


DISTRIBUTOR AGREEMENT
Distributor
Electric City Corp.
Electric City of Illinois L.L.C.
Company
Electric City of Illinois LLC
7th day of September, 1999.
The term of this  Agreement  shall be ten (10)                            years (the "Term")  which shall  commence on the date                            upon which the Company  delivers to  Distributor  the                            last Sample, as defined  hereinafter.
Unless  earlier   terminated   otherwise  provided                   therein,  this  Agreement,  subject to the  commencement  date                   established  in Section 1.3,  shall be effective  immediately.
The term of this  Agreement  shall be ten (10)                            years (the "Term")  which shall  commence on the date                            upon which the Company  delivers to  Distributor  the                            last Sample, as defined  hereinafter.
If Distributor                            complies with all of the terms of this Agree

In [13]:
import pathlib
import json
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B")
dir_path = pathlib.Path("../data/cuad_by_category_2048_all")
NO_ANS_TOKEN = ""
for file in dir_path.glob("*.jsonl"):
    unique_ids = set() # As in original, add is commented out later
    max_tokens = 0
    line_count = 0
    empty_target_count = 0
    non_empty_target_count = 0
    with open(file, 'r') as f:
        for line in f:
            line_count += 1
            data = json.loads(line)
            # unique_ids.add(data['id']) # This line remains commented as in the original selection

            # Count tokens in input field
            tokens = len(tok.encode(data['input']))
            max_tokens = max(max_tokens, tokens)

            # Count empty (just []) vs non-empty targets
            target_value = data.get('target') # Use .get() in case 'target' key is missing

            if target_value == json.dumps([NO_ANS_TOKEN], ensure_ascii=False): # Specifically check for an empty list
                empty_target_count += 1
            else: # All other cases (e.g., non-empty list, dict, None, other types) are considered non-empty
                non_empty_target_count += 1
                
    # Updated print statement to include new counts and use accumulated line_count
    # len(unique_ids) will be 0 as unique_ids.add() is commented out, matching original implied behavior.
    if 'train' in file.name:
        print(f"{file.name}: {len(unique_ids)} unique IDs, {line_count} total lines, max tokens: {max_tokens}, empty_targets (just []): {empty_target_count}, non_empty_targets: {non_empty_target_count}")

Termination_&_Control_Rights_train.jsonl: 0 unique IDs, 404 total lines, max tokens: 2345, empty_targets (just []): 0, non_empty_targets: 404
Metadata_train.jsonl: 0 unique IDs, 404 total lines, max tokens: 2399, empty_targets (just []): 0, non_empty_targets: 404
Legal_Protections_&_Liability_train.jsonl: 0 unique IDs, 404 total lines, max tokens: 2505, empty_targets (just []): 0, non_empty_targets: 404
other_train.jsonl: 0 unique IDs, 0 total lines, max tokens: 0, empty_targets (just []): 0, non_empty_targets: 0
Competition_&_Exclusivity_train.jsonl: 0 unique IDs, 404 total lines, max tokens: 2629, empty_targets (just []): 0, non_empty_targets: 404
Financial_&_Commercial_Terms_train.jsonl: 0 unique IDs, 404 total lines, max tokens: 2360, empty_targets (just []): 0, non_empty_targets: 404
Intellectual_Property_&_Licensing_train.jsonl: 0 unique IDs, 404 total lines, max tokens: 2584, empty_targets (just []): 0, non_empty_targets: 404


In [11]:
import pathlib
import json
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")
dir_path = pathlib.Path("../data/jsonl_stage1")
NO_ANS_TOKEN = "<no_answer>"
for file in dir_path.glob("*.jsonl"):
    unique_ids = set() # As in original, add is commented out later
    max_tokens = 0
    line_count = 0
    empty_target_count = 0
    non_empty_target_count = 0
    with open(file, 'r') as f:
        for line in f:
            line_count += 1
            data = json.loads(line)
            # unique_ids.add(data['id']) # This line remains commented as in the original selection

            # Count tokens in input field
            tokens = len(tok.encode(data['input']))
            max_tokens = max(max_tokens, tokens)

            # Count empty (just []) vs non-empty targets
            target_value = data.get('target') # Use .get() in case 'target' key is missing

            if target_value == {}: # Specifically check for an empty list
                empty_target_count += 1
            else: # All other cases (e.g., non-empty list, dict, None, other types) are considered non-empty
                non_empty_target_count += 1
                
    # Updated print statement to include new counts and use accumulated line_count
    # len(unique_ids) will be 0 as unique_ids.add() is commented out, matching original implied behavior.
    if 'train' in file.name:
        print(f"{file.name}: {len(unique_ids)} unique IDs, {line_count} total lines, max tokens: {max_tokens}, empty_targets (just []): {empty_target_count}, non_empty_targets: {non_empty_target_count}")

train_financial_commercial_terms.jsonl: 0 unique IDs, 532 total lines, max tokens: 16709, empty_targets (just []): 151, non_empty_targets: 381
train_competition_exclusivity.jsonl: 0 unique IDs, 532 total lines, max tokens: 16734, empty_targets (just []): 200, non_empty_targets: 332
train_metadata.jsonl: 0 unique IDs, 532 total lines, max tokens: 16691, empty_targets (just []): 0, non_empty_targets: 532
train_termination_control_rights.jsonl: 0 unique IDs, 532 total lines, max tokens: 16707, empty_targets (just []): 79, non_empty_targets: 453
train_legal_protections_liability.jsonl: 0 unique IDs, 532 total lines, max tokens: 16720, empty_targets (just []): 110, non_empty_targets: 422
train_intellectual_property.jsonl: 0 unique IDs, 532 total lines, max tokens: 16740, empty_targets (just []): 194, non_empty_targets: 338


In [1]:
pip install hf_xet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 13.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer


model_name = "Qwen/Qwen3-1.7B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

thinking content: 
content: Large language models (LLMs) are advanced AI systems designed to understand and generate human-like text. They are trained on vast amounts of data to learn patterns, generate coherent responses, and perform tasks such as writing, coding, answering questions, and more. LLMs are trained using deep learning techniques, often involving neural networks with millions of parameters. These models can adapt and improve over time through iterative training and fine-tuning. They are widely used in various applications, from customer service chatbots to creative writing and scientific research.


In [3]:
# 1) List what your adapter folder thinks it added:
import json
added = json.load(open("../src/stage1/agents/competition_exclusivity/added_tokens.json", "r", encoding="utf-8"))
print(f"Found {len(added)} added tokens:")
for tok, info in added.items():
    print(f"  {tok!r}  →  id {info}")

# 2) Load the same tokenizer and double-check its special_tokens_map:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("../src/stage1/agents/competition_exclusivity", trust_remote_code=True)
print("\nTokenizer.additional_special_tokens:", tok.additional_special_tokens)
print("Tokenizer.vocab_size:", tok.vocab_size)


Found 28 added tokens:
  '</think>'  →  id 151668
  '</tool_call>'  →  id 151658
  '</tool_response>'  →  id 151666
  '<cls_ans?>'  →  id 151670
  '<no_answer>'  →  id 151669
  '<think>'  →  id 151667
  '<tool_call>'  →  id 151657
  '<tool_response>'  →  id 151665
  '<|box_end|>'  →  id 151649
  '<|box_start|>'  →  id 151648
  '<|endoftext|>'  →  id 151643
  '<|file_sep|>'  →  id 151664
  '<|fim_middle|>'  →  id 151660
  '<|fim_pad|>'  →  id 151662
  '<|fim_prefix|>'  →  id 151659
  '<|fim_suffix|>'  →  id 151661
  '<|im_end|>'  →  id 151645
  '<|im_start|>'  →  id 151644
  '<|image_pad|>'  →  id 151655
  '<|object_ref_end|>'  →  id 151647
  '<|object_ref_start|>'  →  id 151646
  '<|quad_end|>'  →  id 151651
  '<|quad_start|>'  →  id 151650
  '<|repo_name|>'  →  id 151663
  '<|video_pad|>'  →  id 151656
  '<|vision_end|>'  →  id 151653
  '<|vision_pad|>'  →  id 151654
  '<|vision_start|>'  →  id 151652

Tokenizer.additional_special_tokens: ['<no_answer>', '<cls_ans?>']
Tokenizer.vocab_

In [20]:
from transformers import AutoTokenizer, AutoConfig

# 1) Load *local* tokenizer from your model_dir
tok_local = AutoTokenizer.from_pretrained(
    "../src/stage1/agents/competition_exclusivity",
    trust_remote_code=True
)
vocab = tok_local.get_vocab()
print("local vocab size:", len(tok_local.get_vocab()))

# 2) Load *hub* config for Qwen3-1.7B
cfg_hub = AutoConfig.from_pretrained(
    "Qwen/Qwen3-1.7B",
    trust_remote_code=True
)
print("hub vocab size:", cfg_hub.vocab_size)

local vocab size: 151669
hub vocab size: 151936
